# CryptoLab 11 — Cryptocurrency Cryptography

**CYBR 3570 Cryptography**

This lab explores cryptographic mechanisms that appear in cryptocurrency and blockchain systems. The goal is **not** to build a real cryptocurrency or wallet. Instead, you will use small, inspectable examples to connect familiar primitives—hashes, signatures, key derivation, secret sharing, and proof systems—to the systems discussed in the chapter.

> **Important:** Several examples intentionally use tiny numbers or simplified constructions so that you can see the mathematics. Do not use these toy constructions to protect real assets or secrets.


## Learning Objectives

By the end of this notebook, you should be able to:

- build a Merkle tree and verify a Merkle inclusion proof;
- model a proof-of-work search;
- use HMAC as a deterministic key-derivation building block;
- sign and verify a transaction-like message with Ed25519;
- explain why canonical encoding matters before hashing or signing;
- create and reconstruct simple additive and Shamir-style secret shares;
- model the algebra behind a Schnorr proof of knowledge;
- explain what cryptography does **not** solve in cryptocurrency systems.


In [1]:
import hashlib
import hmac
import json
import secrets
from dataclasses import dataclass
from typing import Iterable, List, Tuple

print("Notebook ready.")


Notebook ready.


# Part 1 — Hashing Transaction Data

A transaction identifier is often derived from a hash of a canonical byte representation of the transaction. If two implementations serialize the same logical transaction differently, they may compute different hashes.


In [2]:
transaction = {
    "sender": "alice",
    "recipient": "bob",
    "amount": 25,
    "nonce": 7,
}

# Canonicalize the representation: sorted keys and no unnecessary whitespace.
encoded = json.dumps(transaction, sort_keys=True, separators=(",", ":")).encode()
tx_hash = hashlib.sha256(encoded).hexdigest()

print("Canonical bytes:", encoded)
print("Transaction hash:", tx_hash)


Canonical bytes: b'{"amount":25,"nonce":7,"recipient":"bob","sender":"alice"}'
Transaction hash: 59783ad97f994b06939471103c7d221d670b9179aee1374b098a34adc63f8a13


### Exercise 1 — Encoding matters

Create two different pairs of strings `(A, B)` that produce the same raw concatenation `A + B`. Then show why hashing the raw concatenation does not distinguish the two logical inputs.

Next, create an unambiguous encoding using a length prefix or structured encoding.


In [4]:
# TODO: Replace the examples or add your own.
pair1 = ("BAT", "MAN")
pair2 = ("BA", "TMAN")
def encode_pair(a: str, b: str) -> bytes:
    a_len = len(a)
    b_len = len(b)
    return (a+str(a_len)+b+str(b_len)).encode()

raw1 = encode_pair(pair1[0], pair1[1])
raw2 = encode_pair(pair2[0], pair2[1])

print(raw1, raw2, raw1 == raw2)
print(hashlib.sha256(raw1).hexdigest())
print(hashlib.sha256(raw2).hexdigest())



b'BAT3MAN3' b'BA2TMAN4' False
9f734cced763e4a9ea194abc52615afb9ae6d85c5d867a929f00986f39bb731c
c770c04e337e25a577b6e63d0d16665ada1fabc376ffff8a19d4a6fd083158e7


# Part 2 — Merkle Trees

A Merkle tree compresses a list of leaves into one root hash. More importantly, it supports compact proofs that a specific value is included in the tree.


In [5]:
def H(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()


def merkle_parent(left: bytes, right: bytes) -> bytes:
    return H(left + right)


def merkle_root(items: Iterable[bytes]) -> bytes:
    level = [H(item) for item in items]
    if not level:
        raise ValueError("Need at least one item")

    while len(level) > 1:
        if len(level) % 2 == 1:
            level.append(level[-1])
        level = [merkle_parent(level[i], level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]

transactions = [
    b"Alice->Bob:5",
    b"Bob->Carol:2",
    b"Carol->Dave:1",
    b"Dave->Alice:3",
]

root = merkle_root(transactions)
print("Merkle root:", root.hex())


Merkle root: 82c28fd437443b49f29888063d14def0e80a56af3318cd577a173dee94fe081e


### Exercise 2 — Build and verify a Merkle proof

For the four-transaction tree above, build a proof that `Bob->Carol:2` is present. You should need only the sibling hash of Alice's transaction and the hash of the entire right subtree.

Complete the code below so that the recomputed root matches `root`.


In [6]:
leaf_a = H(transactions[0])
leaf_b = H(transactions[1])
leaf_c = H(transactions[2])
leaf_d = H(transactions[3])

right_subtree = merkle_parent(leaf_c, leaf_d)
left_subtree = merkle_parent(leaf_a, leaf_b)

# TODO: Start from leaf_b and use leaf_a plus right_subtree to reconstruct root.
computed_root = merkle_parent(left_subtree, right_subtree)

print("Expected:", root.hex())
print("Computed:", computed_root.hex() if computed_root else "TODO")


Expected: 82c28fd437443b49f29888063d14def0e80a56af3318cd577a173dee94fe081e
Computed: 82c28fd437443b49f29888063d14def0e80a56af3318cd577a173dee94fe081e


# Part 3 — Proof of Work as a Hash Search

In a proof-of-work system, finding a qualifying nonce may require many attempts, but checking a proposed nonce requires only one hash computation.

We will use a tiny target: find a SHA-256 digest beginning with a chosen number of hexadecimal zeroes.


In [7]:
def mine(prefix: bytes, zero_hex_digits: int = 4) -> tuple[int, str, int]:
    target = "0" * zero_hex_digits
    attempts = 0
    nonce = 0
    while True:
        digest = hashlib.sha256(prefix + str(nonce).encode()).hexdigest()
        attempts += 1
        if digest.startswith(target):
            return nonce, digest, attempts
        nonce += 1

nonce, digest, attempts = mine(b"demo-block|", zero_hex_digits=3)
print("Nonce:", nonce)
print("Digest:", digest)
print("Attempts:", attempts)


Nonce: 123
Digest: 000c20076aef27a753f9a72dc717e151682e98e997526007650a0a71a009f3c3
Attempts: 124


### Exercise 3 — Difficulty

Run the mining function with 2, 3, and 4 leading hexadecimal zeroes.

Record the number of attempts for each run. Why should the number of required attempts rise dramatically even though verification remains one hash?


In [10]:
results = {}
for difficulty in [2, 3, 4]:
    # TODO: Call mine() and store the attempts.
    nonce, digest, attempts = mine(b"demo-block|", difficulty)
    print("Difficulty:", difficulty)
    print("Nonce:", nonce)
    print("Digest:", digest)
    print("Attempts:", attempts)





Difficulty: 2
Nonce: 123
Digest: 000c20076aef27a753f9a72dc717e151682e98e997526007650a0a71a009f3c3
Attempts: 124
Difficulty: 3
Nonce: 123
Digest: 000c20076aef27a753f9a72dc717e151682e98e997526007650a0a71a009f3c3
Attempts: 124
Difficulty: 4
Nonce: 25336
Digest: 00009ed586b04d5332212a426a34a62ffd4aa523b1f0713dcf7d47aab7a5c62a
Attempts: 25337


{}

# Part 4 — Deterministic Key Derivation

Hierarchical wallets derive many keys from a smaller amount of master secret material. Production wallet standards have carefully specified derivation rules. Here we will only model the idea that a pseudorandom function can deterministically derive distinct child secrets from one master seed and separate labels.


In [11]:
master_seed = secrets.token_bytes(32)


def derive_child(seed: bytes, label: bytes, index: int) -> bytes:
    msg = label + b"|" + index.to_bytes(4, "big")
    return hmac.new(seed, msg, hashlib.sha512).digest()[:32]

for i in range(3):
    print(i, derive_child(master_seed, b"CYBR3570-demo", i).hex())


0 a973cdd32808d8f73ef953678c902739dcda06bcbeab0fc4d6ed09d6f3c0aa3c
1 4201c60df62782837da8d261e9e847ead0640e5cec6dfaf6dd3ed06a05fafaad
2 809a56bcca0be78b3046dcb7e0bdc6c8bd32129884153cf6053ea3c8cb1a99f8


### Exercise 4 — Domain separation in key derivation

Derive child key `0` twice using two different application labels, such as `b"wallet-signing"` and `b"backup-encryption"`.

1. Are the derived keys the same?
- No
2. Why is using different labels helpful even when the same master secret is used?
- The label is added to the message before hashing adding randomness to it.


In [12]:
# TODO
signing_key_material = derive_child(master_seed, b"wallet-signing", 0)
backup_key_material = derive_child(master_seed, b"backup-encryption", 0)

print("Signing:", signing_key_material.hex() if signing_key_material else "TODO")
print("Backup: ", backup_key_material.hex() if backup_key_material else "TODO")


Signing: 12e12479e074d0d2fc33ab57203ac95f0cb1483a526de9a5e444fb944cd55fcc
Backup:  fd8b203e538dfd86e8f3efd9095d3467229280b67fa1e0a721310a1c04b2eb7d


# Part 5 — Signing a Transaction-Like Message

Cryptocurrency transactions are authorized with digital signatures, not by encrypting the transaction. Anyone who knows the public key can verify that the holder of the corresponding private key approved the signed bytes.

We will use Ed25519 from the `cryptography` package.


In [18]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature

private_key = Ed25519PrivateKey.generate()
public_key = private_key.public_key()

payment = {
    "from": "alice",
    "to": "bob",
    "amount": 10,
    "nonce": 42,
}
message = json.dumps(payment, sort_keys=True, separators=(",", ":")).encode()
signature = private_key.sign(message)

try:
    public_key.verify(signature, message)
    print("Signature valid")
except InvalidSignature:
    print("Signature invalid")


Signature valid


### Exercise 5 — Tamper with the transaction

Change the amount from `10` to `1000` **after** the signature is created. Serialize the modified transaction in the same canonical way and verify the original signature against the modified bytes.

What happens, and which security property is being demonstrated?
- This shows authenticity and the validity of a message, this check makes it difficult for attackers to change messages without the receiver noticing.


In [19]:
tampered = dict(payment)
tampered["amount"] = 1000
modified_message = json.dumps(tampered, sort_keys=True, separators=(",", ":")).encode()

# TODO: Verify signature against modified_message and catch InvalidSignature.
try:
    public_key.verify(signature, modified_message)
    print("Signature valid")
except InvalidSignature:
    print("Signature invalid")

Signature invalid


# Part 6 — Additive Secret Sharing

Secret sharing changes *where* risk is concentrated. Instead of keeping one secret in one place, we can split it into shares.

The simplest construction is additive sharing modulo some number.


In [20]:
def additive_split(secret: int, n: int, modulus: int) -> List[int]:
    if not 0 <= secret < modulus:
        raise ValueError("secret must be in the field/ring")
    shares = [secrets.randbelow(modulus) for _ in range(n - 1)]
    final_share = (secret - sum(shares)) % modulus
    return shares + [final_share]


def additive_recover(shares: Iterable[int], modulus: int) -> int:
    return sum(shares) % modulus

secret = 47
shares = additive_split(secret, 4, 101)
print("Shares:", shares)
print("Recovered:", additive_recover(shares, 101))


Shares: [6, 74, 48, 20]
Recovered: 47


### Exercise 6 — Missing a share

Remove one share and try to reconstruct by simply adding the remaining values.

1. Do you recover the original secret?
- No
2. Why does this simple scheme require all shares?
- If they are not all added together to form a whole then the end value will be incorrect.
3. How is that different from a threshold secret-sharing scheme?
- Threshold doesn't need all of the shares while this needs all of them.


In [21]:
# TODO
partial = shares[:-1]
print("Using only:", partial)
print("Partial sum:", additive_recover(partial, 101))


Using only: [6, 74, 48]
Partial sum: 27


# Part 7 — A Small Shamir Secret-Sharing Demonstration

For a 2-of-3 scheme, use a random line:

`f(x) = secret + a*x (mod p)`

Any two points determine the line, so any two shares recover the secret. One point alone does not.


In [22]:
P = 2089  # small prime for the classroom demo


def inv(x: int, p: int) -> int:
    return pow(x, -1, p)


def shamir_2_of_n_split(secret: int, n: int, p: int = P) -> List[Tuple[int, int]]:
    a = secrets.randbelow(p)
    return [(x, (secret + a * x) % p) for x in range(1, n + 1)]


def recover_from_two(s1: Tuple[int, int], s2: Tuple[int, int], p: int = P) -> int:
    x1, y1 = s1
    x2, y2 = s2
    # Evaluate the interpolating line at x = 0.
    term1 = y1 * (-x2) * inv(x1 - x2, p)
    term2 = y2 * (-x1) * inv(x2 - x1, p)
    return (term1 + term2) % p

shamir_secret = 1337
shamir_shares = shamir_2_of_n_split(shamir_secret, 3)
print("Shares:", shamir_shares)
print("Recover 1+2:", recover_from_two(shamir_shares[0], shamir_shares[1]))
print("Recover 1+3:", recover_from_two(shamir_shares[0], shamir_shares[2]))


Shares: [(1, 1432), (2, 1527), (3, 1622)]
Recover 1+2: 1337
Recover 1+3: 1337


### Exercise 7 — Threshold reasoning

Demonstrate that each pair of the three shares recovers the same secret.

Then explain why one share by itself is insufficient to determine which of many possible lines—and therefore which secret—was used.
- The Function requires two

In [23]:
# TODO: Verify all three pairs.
print(recover_from_two(shamir_shares[0], shamir_shares[1]))
print(recover_from_two(shamir_shares[0], shamir_shares[2]))
print(recover_from_two(shamir_shares[1], shamir_shares[2]))


1337
1337
1337


# Part 8 — Schnorr-Style Proof of Knowledge (Toy Group)

A zero-knowledge proof can demonstrate knowledge of a secret without sending the secret itself.

We will model the algebra in a tiny multiplicative group modulo a prime. This is **only** for understanding the protocol.

Let the prover know secret `a`, with public value:

`A = g^a mod p`

The prover will:

1. choose random `r` and send `R = g^r`;
2. receive challenge `c`;
3. send `s = r + c*a`;
4. verifier checks `g^s == R * A^c (mod p)`.


In [24]:
p = 23
q = 11  # subgroup order
g = 2   # generator of a subgroup of order 11 modulo 23

a = 7
A = pow(g, a, p)

r = secrets.randbelow(q)
R = pow(g, r, p)
c = secrets.randbelow(q)
s = (r + c * a) % q

left = pow(g, s, p)
right = (R * pow(A, c, p)) % p

print("Public A:", A)
print("Commitment R:", R)
print("Challenge c:", c)
print("Response s:", s)
print("Verification:", left == right)


Public A: 13
Commitment R: 1
Challenge c: 8
Response s: 1
Verification: True


### Exercise 8 — What is and is not revealed?

Answer in Markdown:

1. Which values are public in the transcript?
- Public A value, R value, Challenge c, Response s, g, p
2. Which value must remain secret?
- secret a, and r
3. What would go wrong if the prover reused the same `r` with two different challenges?
- It would work but the value r could become known.
4. Which ZKP property would be broken if a prover could create valid transcripts without knowing `a`?
- The point is to show you know without revealing, undermine the entire point.


# Part 9 — Fiat–Shamir: Replacing the Interactive Challenge

In a noninteractive version, the challenge can be derived from the transcript context using a hash. The crucial lesson is that the hash must bind all security-relevant values.


In [25]:
def challenge_from_context(g: int, R: int, A: int, q: int) -> int:
    data = f"g={g}|R={R}|A={A}".encode()
    return int.from_bytes(hashlib.sha256(data).digest(), "big") % q

r2 = secrets.randbelow(q)
R2 = pow(g, r2, p)
c2 = challenge_from_context(g, R2, A, q)
s2 = (r2 + c2 * a) % q

check = pow(g, s2, p) == (R2 * pow(A, c2, p)) % p
print("Noninteractive proof verifies:", check)


Noninteractive proof verifies: True


### Exercise 9 — Context binding

Suppose a developer changes the challenge to depend only on `A` and omits `R`.

Explain why that is dangerous. You do not need to implement the attack; focus on the protocol-design principle:

> Why should the challenge depend on the commitment and other relevant context?
- R adds randomness to it and makes the challenge independent of the provers commitment. Removing it would invalidate that property. This could let an attacker validate without showing proof of knowledge.

# Part 10 — System Security Reflection

Cryptocurrency systems combine cryptographic primitives with software, networking, key custody, human processes, and economic incentives.

For each scenario below, identify the **primary failure layer**:

- cryptographic primitive
- protocol design
- randomness/key generation
- encoding/domain separation
- implementation
- key management/operations

### Scenarios

1. A wallet generates all master seeds from only 32 bits of entropy.
- Randomness/key generation
2. A multisignature implementation accepts a rogue public key without proof of possession.
- Protocol design
3. Two logical protocol inputs serialize to the same byte string before hashing.
- encoding/domain separation
4. An otherwise secure threshold-signature paper is implemented but a required parameter-validation check is omitted.
- Implementation
5. An organization stores every threshold share in the same cloud administrator account.
- key managment/operations

Write your classifications and one-sentence justifications below.


## Final Reflection

Answer each question in 2–4 sentences.

1. **Hashing vs. signatures:** Why is a transaction hash not enough to prove who authorized a transaction?
- It proves that it wasn't edited in transit, but that doesn't say who authorized or sent it. Signing just proves who the sender is and that they are legitimate, while hashing verifies it wasn't edited.
2. **Merkle trees:** What does a Merkle proof allow a verifier to learn or verify without processing an entire dataset?
- I can prove a transaction is included in the dataset without recomputing the entire hash. It lets you use 3 hashes to compute rather than potentially thousands.
3. **Key management:** Why does deterministic or threshold key management reduce some risks while creating new operational responsibilities?
- I can derive multiple keys from the same seed. this lets keys be regenerated when needed, but you need to store the seed as a secret. the challenge is to avoid reusing the same keys, or the same info/parameters, and if the seed is lost the keys could be generated.
4. **Zero knowledge:** Give one cybersecurity use case outside cryptocurrency where proving a fact without revealing the underlying secret could be valuable.
- proof od identity with banking where you do not want to share your PI. Or age verification.
5. **Course theme:** Choose one chapter failure and explain how it demonstrates that cryptographic security is a system property rather than an c property.
- Weak seeds, if a seed is weak where an attacker could derive it then one of the most important fundamental cryptographic principal of randomness would be void. If a system cannot generate good PRN then an algorithm wouldn't have a chance to fail if its randomness used for its basis ins't secure.
